In [0]:
catalog = "ds_mini_project_catalog"   # your catalog name
schema = "powerbi_schema"             # your schema name

# Step 1: Create catalog if not exists
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# Step 2: Use the catalog
spark.sql(f"USE CATALOG {catalog}")

# Step 3: Create schema inside the catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

# Step 4: Use the schema
spark.sql(f"USE {schema}")

In [0]:
import pandas as pd
import os
from collections import defaultdict

base_path = "../data/07_powerbi_1"
files = os.listdir(base_path)

def make_unique(cols):
    """Ensure column names are fully unique"""
    seen = defaultdict(int)
    new_cols = []

    for c in cols:
        if seen[c] == 0:
            new_cols.append(c)
        else:
            new_cols.append(f"{c}_{seen[c]}")
        seen[c] += 1

    return new_cols

for file in files:
    if file.endswith(".csv"):
        file_path = os.path.join(base_path, file)

        print(f"\nProcessing file: {file}")

        # 1. Read CSV
        df = pd.read_csv(file_path)

        # 2. Drop unnamed
        df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

        # 3. Clean column names
        df.columns = df.columns.str.replace(r"[ ,;{}()\n\t=]", "_", regex=True)

        # 4. FORCE UNIQUE NAMES (Pandas level)
        df.columns = make_unique(df.columns)

        # 5. FINAL SAFETY CHECK (IMPORTANT)
        # Spark fails if ANY duplicates exist
        if df.columns.duplicated().any():
            print("Still duplicate columns detected - fixing again")
            df.columns = make_unique(df.columns)

        # 6. Convert AFTER cleaning
        sdf = spark.createDataFrame(df)

        # 7. Table name
        table_name = file.replace(".csv", "").lower()
        full_table_name = f"ds_mini_project_catalog.powerbi_schema.{table_name}"

        print(f"Writing to: {full_table_name}")
        print(f"Rows: {sdf.count()}, Columns: {len(sdf.columns)}")

        # 8. Drop table
        spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")

        # 9. Write Delta
        sdf.write \
            .mode("overwrite") \
            .format("delta") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_table_name)

        print(f"Successfully written: {full_table_name}")